In [ ]:
# IMAGE
# TN THIS ALL OVER CODE WE LEARN HOW TO EXTRACT WHAT IS  WHAT IS IN THE IMAGE



# importing the modules that are needed
import cv2
import numpy as np
import argparse
import time

In [ ]:
config='yolov3.cfg' # cfg folder
weight='yolov3.weights'
class_names='coco.name' # data folde

In [ ]:
def load_yolo():
       net = cv2.dnn.readNet(config,weight)
       classes = []
       with open(class_names,"r") as f:
              classes = [line.strip() for line in f.readlines()]
              layer_names = net.getLayerNames()
              output_layers =[layer_names[i[0]-1] for i in net.getUnconnectedOutLayers()]
              colors = np.random.uniform(0,255,size=(len(classes),3))
              return net,classes,colors,output_layers

In [ ]:
def load_image(img_path):
       # image loading
       img = cv2.imread(img_path)
       img = cv2.resize(img,None,fx=0.4,fy=0.4)
       height, weight, channels= img.shape
       return img, height, width, channels

In [ ]:
def detect_objects(image,net,outputLayers):
       blob = cv2.dnn.blobFromImage(image, scalefactor=0.00392, size=(320,320),mean=(0,0,0),
                                    swapRB=True,crop=False)
       net.setInput(blob)
       outputs = net.forward(outputLayers)
       return blob, outputs

In [ ]:
def get_box_dimension(outputs,height,width):
       boxes = []
       confs = []
       class_ids = []
       for output in outputs:
              for detect in output:
                     scores = detect[5:]
                     class_id = np.argmax(scores)
                     conf = scores[class_id]
                     if conf >0.3:
                        center_x=int(detect[0]*width)
                        center_y=int(detect[1]*height)
                        w=int(detect[2]*width)
                        h=int(detect[3]*height)
                        x=int(center_x-w/2)
                        y=int(center_y-h/2)
                        boxes.append([x,y,w,h])
                        confs.append(float(conf))
                        class_ids.append(class_id)
       return boxes,confs,class_ids

In [ ]:
def draw_labels(boxes,confs,colors,class_ids,classes,img):
   indexs = cv2.dnn.NMSBoxes(boxes,confs,.5,.4)
   font = cv2.FONT_HERSHEY_PLAIN
   for i in range(len(boxes)):
      if i in indexs:
         x,y,w,h = boxes[i]
         label = str(classes[class_ids[i]])
         conf = str(round(confs[i],2))
         color = colors[i]
         cv2.rectangle(img,(x,y),(x+w,y+h),color,2)
         cv2.putText(img,label,(x,y,-5),font,1,color,1)
  cv2.imshow("image",img)

In [ ]:
def image_detect(img_path):
   model,classes,colors,output_layers=load_yolo()
   image,height,width,channels=load_image(img_path)
   blob,output=detect_objects(image,model,output_layers)
   boxes,confs,class_ids=get_box_dimension(output,height,width)
   draw_labels(boxes,confs,colors,class_ids,classes,image)
   while True:
     key = cv2.waitKey(1)
     if key == 27:
        break
   cv2.destroyAllWindows()

In [ ]:
image = 'bicycle.jpg'
image_detect(image)

error: OpenCV(4.13.0) /io/opencv/modules/dnn/src/darknet/darknet_importer.cpp:210: error: (-212:Parsing error) Failed to open NetParameter file: yolov3.cfg in function 'readNetFromDarknet'
